In [1]:
pip install selenium



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from selenium import webdriver


In [1]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import requests
import time
import os

save_dir = os.path.expanduser("~/Downloads/harpersbazaar_images")
os.makedirs(save_dir, exist_ok=True)

driver = webdriver.Chrome()
driver.get("https://www.harpersbazaar.com/celebrity/red-carpet-dresses/a71204913/every-look-red-carpet-photos-met-gala-2026/")
time.sleep(5)

images = driver.find_elements(By.TAG_NAME, "img")

count = 0

for img in images:
    try:
        url = img.get_attribute("src")

        # Filter out junk
        if url and ("jpg" in url or "jpeg" in url or "webp" in url):

            response = requests.get(url)

            if response.status_code == 200:
                filename = os.path.join(save_dir, f"harpers_image_{count}.jpg")

                with open(filename, "wb") as f:
                    f.write(response.content)

                print(f"Saved {count}")
                count += 1

    except:
        pass

driver.quit()

print(f"Downloaded {count} images")

Saved 0
Saved 1
Saved 2
Saved 3
Saved 4
Saved 5
Saved 6
Saved 7
Saved 8
Saved 9
Saved 10
Saved 11
Saved 12
Saved 13
Saved 14
Saved 15
Saved 16
Saved 17
Saved 18
Saved 19
Saved 20
Saved 21
Saved 22
Saved 23
Saved 24
Saved 25
Saved 26
Saved 27
Saved 28
Saved 29
Saved 30
Saved 31
Saved 32
Downloaded 33 images


In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time, json, re, os

save_dir = os.path.expanduser("~/Downloads/harpersbazaar_images")
os.makedirs(save_dir, exist_ok=True)

driver = webdriver.Chrome()
driver.get("https://www.harpersbazaar.com/celebrity/red-carpet-dresses/a71204913/every-look-red-carpet-photos-met-gala-2026/")
time.sleep(8)

# Scroll down to load all content
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

data = []

slides = driver.find_elements(By.CSS_SELECTOR, "section[data-anchor-id]")

for i, slide in enumerate(slides):
    try:
        name = slide.find_element(By.TAG_NAME, "h2").text.strip()

        paragraphs = slide.find_elements(By.TAG_NAME, "p")
        designer_text = " ".join([p.text.strip() for p in paragraphs if p.text.strip()])

        match = re.search(r'\bin\b (.+)', designer_text, re.IGNORECASE)
        designer = match.group(1).strip() if match else "unknown"

        data.append({
            "index": i,
            "celebrity": name,
            "caption": designer_text,
            "designer": designer
        })

        print(f"{i}: {name} → {designer}")

    except:
        pass

with open(os.path.join(save_dir, "designers.json"), "w") as f:
    json.dump(data, f, indent=2)

driver.quit()
print(f"\nScraped {len(data)} entries")
print(f"Saved to {save_dir}/designers.json")

0: Beyoncé → Custom Olivier Rousteing
1: Rihanna → Maison Margiela FW25 Couture and Briony Raymond and Dyne jewelry
2: Blue Ivy Carter → Balenciaga
3: Kendall Jenner → Custom Gap Studio by Zac Pose and Buccellati Jewelry
4: Bad Bunny → Custom Zara
5: Teyana Taylor → Custom Tom Ford by Haider Ackermann
6: Chase Infiniti → Thom Browne
7: Madonna → Saint Laurent
8: Jennie → Chanel
9: Kim Kardashian → Custom Allen Jones and Whitaker Malem
10: A$AP Rocky → Chanel
11: Doechii → Custom Marc Jacobs
12: Lisa → Robert Wun and Bulgari jewelry
13: Kylie Jenner → Custom Schiaparelli
14: Hailey Bieber → Custom Saint Laurent
15: Ayo Edebiri → Chanel
16: Sabrina Carpenter → Custom Christian Dior
17: Alexa Chung → Dior
18: Jay-Z → Louis Vuitton
19: Odessa A'zion → Custom Valentino and Pandora Jewelry
20: Tate McRae → Ludovic de Saint Sernin
21: Blake Lively → Versace, Lorraine Schwartz jewelry, and Judith Leiber bag
22: Tessa Thompson → Valentino and Pandora Jewelry
23: Cher → Burberry and Swarovski je

In [26]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import requests
import time
import os
import json
import re
import hashlib

# Save folder
save_dir = os.path.expanduser("~/Downloads/vogue_images")
os.makedirs(save_dir, exist_ok=True)

# Start browser
driver = webdriver.Chrome()
driver.get("https://www.vogue.co.uk/gallery/met-gala-2026-red-carpet-fashion")
time.sleep(10)

saved_urls = set()
saved_hashes = set()
count = 0
data = []

def scrape_current_images():
    global count

    images = driver.find_elements(By.TAG_NAME, "img")

    for img in images:
        try:
            # Force lazy-load
            driver.execute_script("arguments[0].scrollIntoView(true);", img)
            time.sleep(0.2)

            url = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-lazy-src")
                or img.get_attribute("srcset")
            )

            if not url:
                continue

            # If srcset, grab highest-res version
            if "," in url:
                url = url.split(",")[-1].split(" ")[0]

            if "media.vogue.co.uk" not in url:
                continue

            if url in saved_urls:
                continue

            # Upgrade resolution
            url = re.sub(r"w_\d+", "w_1280", url)

            response = requests.get(url, timeout=20)

            if response.status_code == 200:
                file_hash = hashlib.md5(response.content).hexdigest()

                if file_hash in saved_hashes:
                    saved_urls.add(url)
                    continue

                filename = os.path.join(save_dir, f"vogue_image_{count}.jpg")

                with open(filename, "wb") as f:
                    f.write(response.content)

                saved_urls.add(url)
                saved_hashes.add(file_hash)

                print(f"Saved image {count}")
                count += 1

        except Exception as e:
            pass


def scrape_captions():
    captions = driver.find_elements(
        By.CSS_SELECTOR,
        "[data-testid='GallerySlideCaptionDekContainer'] p"
    )

    for caption in captions:
        text = caption.text.strip()

        if not text:
            continue

        match = re.search(r'\bin\b (.+)', text, re.IGNORECASE)
        designer = match.group(1).strip().rstrip(".") if match else "unknown"

        name_match = re.match(r'^(.+?)\s+in\s+', text, re.IGNORECASE)
        celebrity = name_match.group(1).strip() if name_match else "unknown"

        entry = {
            "celebrity": celebrity,
            "designer": designer,
            "caption": text
        }

        if entry not in data:
            data.append(entry)
            print(f"Scraped: {celebrity} → {designer}")


# Smart scrolling
last_height = driver.execute_script("return document.body.scrollHeight")
no_change_count = 0

while no_change_count < 8:
    scrape_current_images()
    scrape_captions()

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)

    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        no_change_count += 1
        print(f"No new content ({no_change_count}/8)")
    else:
        no_change_count = 0
        print("Loaded more content")

    last_height = new_height


# Save metadata
with open(os.path.join(save_dir, "designers.json"), "w") as f:
    json.dump(data, f, indent=2)

driver.quit()

print(f"\nDownloaded {count} images")
print(f"Scraped {len(data)} designer entries")
print(f"Saved to {save_dir}")

Saved image 0
Saved image 1
Saved image 2
Saved image 3
Saved image 4
Scraped: Beyoncé → Olivier Rousteing and Chopard
Scraped: Blue Ivy Carter → Balenciaga and Henry & Henry
Scraped: Jay-Z → Louis Vuitton
Scraped: Venus Williams → Swarovski
Loaded more content
Saved image 5
Saved image 6
Saved image 7
Saved image 8
Saved image 9
Saved image 10
Saved image 11
Scraped: Nicole Kidman → Chanel
Scraped: Anna Wintour → Chanel
Scraped: Chloe Malle → Colleen Allen
Scraped: Chioma Nnadi → Steve O Smith
Loaded more content
Saved image 12
Saved image 13
Saved image 14
Saved image 15
Scraped: Kim Kardashian → Allen Jones and Whitaker Malem
Scraped: Teyana Taylor → Tom Ford
Scraped: Bad Bunny → Zara
Scraped: Zoe Kravitz → Saint Laurent and Jessica McCormack
Loaded more content
Saved image 16
Saved image 17
Saved image 18
Saved image 19
Scraped: Charli xcx → Saint Laurent
Scraped: Gwendoline Christie → Giles Deacon and Herbert Levine
Scraped: Margot Robbie → Chanel
Scraped: Daisy Edgar-Jones → McQu

WebDriverException: Message: tab crashed
  (Session info: chrome=147.0.7727.138)
